# Tome 0, Chapter 8: Counting and First Probability (Python companion)

Exercises first, solutions at the end. Computer-science threads: the **birthday attack** on hash functions,
**dynamic programming** for counting, and the $O(n\log n)$ **longest increasing subsequence** algorithm.

In [ ]:
import math
import hashlib
import bisect
import itertools
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## Exercises

**P8.1 (★) Hats and $1/e$.** Count derangements exactly for $n \le 10$ by brute force and with the formula of Proposition 8.3. Then estimate $P(\text{no fixed point})$ for $n = 1000$ by simulating 100 000 random permutations, and plot the histogram of the number of fixed points against the Poisson(1) probabilities $e^{-1}/k!$.

In [ ]:
# Your code here


**P8.2 (★) $\varphi$ by inclusion–exclusion.** Implement Euler's $\varphi$ with the product formula of Exercise 8.4 (factor with `sympy.factorint`) and compare with a direct gcd count and with `sympy.totient` for $n \le 2000$.

In [ ]:
# Your code here


**P8.3 (★★) CS application: the birthday attack.** A hash truncated to $b$ bits has $2^b$ values. By the birthday problem, a collision appears after about $\sqrt{\pi 2^b/2}$ random inputs, not $2^b$. Find collisions of SHA-256 truncated to $b = 16, 24, 32$ bits and compare the number of tries with $\sqrt{\pi 2^{b}/2}$. What does this imply for the length of real hashes?

In [ ]:
# Your code here


**P8.4 (★★) CS application: counting by dynamic programming.** Count the ways to pay 1 euro (100 cents) with coins of 1, 2, 5, 10, 20, 50 cents and 1 euro, using a DP table. Check the answer against the coefficient of $x^{100}$ in $\prod_c \frac{1}{1-x^c}$ (a generating function, Tome 4).

In [ ]:
# Your code here


**P8.5 (★★) Cycles and coupons.** Simulate the number of cycles of random permutations of size 1000 and the coupon collector for $n = 100$, and compare the averages with $H_n$ and $nH_n$.

In [ ]:
# Your code here


**P8.6 (★★★) CS application: longest increasing subsequence.** Implement *patience sorting*: scan the sequence, keeping for each length $\ell$ the smallest possible last element of an increasing subsequence of length $\ell$ (a sorted list updated with `bisect`). This runs in $O(n\log n)$. For random permutations of size $n$, the LIS length is about $2\sqrt n$; check it, and check Erdős–Szekeres on random sequences of length $n^2+1$.

In [ ]:
# Your code here


---
## Solutions

### P8.1

In [ ]:
def derangements_formula(n):
    return round(math.factorial(n) * sum((-1) ** k / math.factorial(k) for k in range(n + 1)))

for n in range(1, 9):
    brute = sum(all(p[i] != i for i in range(n)) for p in itertools.permutations(range(n)))
    assert brute == derangements_formula(n)
print("D_n for n = 1..10:", [derangements_formula(n) for n in range(1, 11)])

n, trials = 1000, 100_000
fixed = np.array([(rng.permutation(n) == np.arange(n)).sum() for _ in range(trials)])
print(f"P(no fixed point) ~ {np.mean(fixed == 0):.4f}   1/e = {1 / math.e:.4f}")
ks = np.arange(8)
plt.bar(ks - 0.2, [np.mean(fixed == k) for k in ks], width=0.4, label="simulation")
plt.bar(ks + 0.2, [math.exp(-1) / math.factorial(k) for k in ks], width=0.4, label="Poisson(1)")
plt.xlabel("number of fixed points"); plt.legend(); plt.show()

### P8.2

In [ ]:
def phi(n):
    r = n
    for p in sp.factorint(n):
        r = r // p * (p - 1)
    return r

assert all(phi(n) == sum(math.gcd(k, n) == 1 for k in range(1, n + 1)) == sp.totient(n) for n in range(1, 2001))
print("phi agrees with the gcd count and sympy.totient for n <= 2000")

### P8.3

In [ ]:
def first_collision(bits, seed=0):
    seen = {}
    for i in itertools.count():
        h = int.from_bytes(hashlib.sha256(f"{seed}-{i}".encode()).digest(), "big") >> (256 - bits)
        if h in seen:
            return i + 1
        seen[h] = i

for bits in [16, 24, 32]:
    tries = np.mean([first_collision(bits, s) for s in range(20)])
    print(f"b = {bits}: average tries {tries:9.0f}   sqrt(pi 2^b / 2) = {math.sqrt(math.pi * 2**bits / 2):9.0f}   2^b = {2**bits}")

A $b$-bit hash offers only about $b/2$ bits of collision resistance. That is why SHA-256 is used where 128-bit security is required, and why MD5 (128 bits, plus structural weaknesses) is broken.

### P8.4

In [ ]:
coins = [1, 2, 5, 10, 20, 50, 100]
ways = [1] + [0] * 100
for c in coins:                      # process coins one by one so each combination is counted once
    for amount in range(c, 101):
        ways[amount] += ways[amount - c]
print("ways to pay 100 cents:", ways[100])

x = sp.symbols("x")
gf = sp.prod([sum(x ** (c * k) for k in range(100 // c + 1)) for c in coins])
print("coefficient of x^100:", sp.Poly(sp.expand(gf), x).coeff_monomial(x**100))

### P8.5

In [ ]:
def n_cycles(p):
    seen, c = np.zeros(len(p), bool), 0
    for i in range(len(p)):
        if not seen[i]:
            c += 1
            j = i
            while not seen[j]:
                seen[j] = True
                j = p[j]
    return c

n = 1000
H = lambda n: sum(1 / k for k in range(1, n + 1))
print(f"cycles: mean {np.mean([n_cycles(rng.permutation(n)) for _ in range(2000)]):.3f}   H_1000 = {H(1000):.3f}")

def coupon(n):
    got, t = set(), 0
    while len(got) < n:
        got.add(int(rng.integers(n))); t += 1
    return t
print(f"coupons: mean {np.mean([coupon(100) for _ in range(2000)]):.1f}   100 H_100 = {100 * H(100):.1f}")

### P8.6

In [ ]:
def lis_length(seq):
    tails = []                       # tails[l] = smallest last element of an increasing subsequence of length l+1
    for x in seq:
        k = bisect.bisect_left(tails, x)
        if k == len(tails):
            tails.append(x)
        else:
            tails[k] = x
    return len(tails)

for n in [10**3, 10**4, 10**5, 10**6]:
    L = lis_length(rng.permutation(n))
    print(f"n = {n:>7}: LIS = {L:5d},  2 sqrt(n) = {2 * math.sqrt(n):7.1f}")

for n in range(2, 30):
    for _ in range(20):
        s = rng.permutation(n * n + 1)
        assert max(lis_length(s), lis_length(-s)) >= n + 1
print("Erdos-Szekeres confirmed on random sequences")

The LIS of a random permutation is $2\sqrt n - c\,n^{1/6}$ with fluctuations of order $n^{1/6}$, following the Tracy–Widom law that also governs the largest eigenvalue of random matrices (Baik, Deift and Johansson, 1999), a surprise that connects Chapter 8 to Tome 11.